In [1]:
# Install the face-recognition library
!pip install -q insightface onnxruntime


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys; sys.path.append("..") 
from src import generate, evaluate, config
import importlib; importlib.reload(generate); importlib.reload(evaluate)

/Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'src.evaluate' from '/Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/notebooks/../src/evaluate.py'>

In [3]:
# Generate a batch of 10 across different seeds, with the good prompt
generate.generate_for_eval(
    n=10,
    prompt="a photo of sks man, color photograph, sharp focus, natural daylight",
    negative_prompt="sepia, grainy, vintage, black and white, blurry, low quality",
    steps=40, guidance_scale=7.0, base_seed=0,
)

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 10.21it/s]
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
100%|██████████| 40/40 [01:43<00:00,  2.59s/it]


Saved 1 image(s) to /Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/outputs/generated


100%|██████████| 40/40 [01:35<00:00,  2.40s/it]


Saved 1 image(s) to /Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/outputs/generated


100%|██████████| 40/40 [01:33<00:00,  2.33s/it]


Saved 1 image(s) to /Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/outputs/generated


... (11 outputs left out)

Saved 1 image(s) to /Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/outputs/generated


100%|██████████| 40/40 [01:51<00:00,  2.79s/it]


Saved 1 image(s) to /Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/outputs/generated
Generated 10 images for evaluation in /Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/outputs/generated


PosixPath('/Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/outputs/generated')

In [4]:
# Measure identity similarity + novelty
res = evaluate.evaluate(train_dir=config.TRAIN_DIR)
res

/Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/venv/lib/python3.12/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'CoreMLExecutionProvider, AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/venv/lib/python3.12/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Generated images:         10
  with a detectable face: 10  (100%)
Real reference faces:     4
--------------------------------------------------------
       gen_000.png:  identity=0.181   nearest-train=0.285
       gen_001.png:  identity=0.227   nearest-train=0.245
       gen_002.png:  identity=0.474   nearest-train=0.503
       gen_003.png:  identity=0.247   nearest-train=0.336
       gen_004.png:  identity=0.348   nearest-train=0.478
       gen_005.png:  identity=0.374   nearest-train=0.428
       gen_006.png:  identity=0.419   nearest-train=0.463
       gen_007.png:  identity=0.213   nearest-train=0.295
       gen_008.png:  identity=0.461   nearest-train=0.531
       gen_009.png:  identity=0.425   nearest-train=0.488
--------------------------------------------------------
Mean identity similarity: 0.337
Best / worst:             0.474 / 0.181
Above threshold 0.35:     5/10 (50%)
Mean nearest-train sim:   0.405 (well below 1.0 = novel, not copied)


{'n_total': 10,
 'n_faces': 10,
 'mean_identity': 0.3370549663901329,
 'best_identity': 0.4742725193500519,
 'worst_identity': 0.18112286925315857,
 'pass_rate': 0.5,
 'threshold': 0.35,
 'similarities': [0.18112286925315857,
  0.22661370038986206,
  0.4742725193500519,
  0.24732381105422974,
  0.3483653962612152,
  0.37380772829055786,
  0.41948333382606506,
  0.21288205683231354,
  0.46142253279685974,
  0.4252557158470154],
 'mean_novelty': 0.40501752495765686}

In [ ]:
generate.generate_for_eval(
    n=10,
    prompt="a photo of sks man",
    negative_prompt=None,
    steps=30, guidance_scale=7.5, base_seed=0,
)
res_plain = evaluate.evaluate(train_dir=config.TRAIN_DIR)
res_plain

In [ ]:
# Evaluate rank 16
generate.generate_for_eval(
    n=10,
    prompt="a photo of sks man, color photograph, sharp focus, natural daylight",
    negative_prompt="sepia, grainy, vintage, black and white, blurry, low quality",
    steps=40, guidance_scale=7.0, base_seed=0,
)
evaluate.evaluate(train_dir=config.TRAIN_DIR)

Generate the baseline images

In [ ]:
import torch, gc
from diffusers import StableDiffusionPipeline
from src import config

base_dir = config.LORA_DIR.parent / "generated_baseline"
base_dir.mkdir(parents=True, exist_ok=True)
for p in base_dir.glob("*.png"):
    p.unlink()

pipe = StableDiffusionPipeline.from_pretrained(config.BASE_MODEL).to(config.DEVICE)
pipe.safety_checker = None

for k in range(10):
    g = torch.Generator(device=config.DEVICE).manual_seed(k)
    img = pipe(
        "a photo of a man, color photograph, sharp focus, natural daylight",
        negative_prompt="sepia, grainy, vintage, black and white, blurry, low quality",
        num_inference_steps=40, guidance_scale=7.0, generator=g,
    ).images[0]
    img.save(base_dir / f"base_{k:03d}.png")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("baseline images saved to", base_dir)

Score them

In [ ]:
from src import evaluate, config
evaluate.evaluate(generated_dir=base_dir, train_dir=config.TRAIN_DIR)

A real-photo ceiling.


In [ ]:
import numpy as np
from src import evaluate, config

test_embs = evaluate._embeddings_for_dir(config.TEST_DIR)
train_embs = evaluate._embeddings_for_dir(config.TRAIN_DIR)
train_mat = np.stack(list(train_embs.values()))

sims = []
for name, e in test_embs.items():
    mean_sim = float(np.mean(train_mat @ e))
    best_sim = float(np.max(train_mat @ e))
    sims.append(mean_sim)
    print(f"{name}: mean-to-train={mean_sim:.3f}  best-match={best_sim:.3f}")

print("-" * 40)
print(f"Real test-vs-train MEAN identity (ceiling): {np.mean(sims):.3f}")

Variance

In [ ]:
import numpy as np
runs = {
  "baseline":       [-0.056,-0.006,-0.005,0.115,0.008,-0.006,0.053,0.007],
  "rank4_quality":  [0.206,0.349,0.233,0.322,0.245,0.250,0.247,0.269],
  "rank4_plain":    [0.206,0.051,0.259,0.103,0.255,0.054,0.254,0.139,0.224],
  "rank16_quality": [0.181,0.227,0.474,0.247,0.348,0.374,0.419,0.213,0.461,0.425],
}
for name, v in runs.items():
    v = np.array(v)
    print(f"{name:16s} mean={v.mean():.3f}  std={v.std():.3f}  n={len(v)}")

Overfitting / language-drift check

In [ ]:
import torch, gc
from diffusers import StableDiffusionPipeline
from src import config, evaluate

drift_dir = config.LORA_DIR.parent / "generated_drift"
drift_dir.mkdir(parents=True, exist_ok=True)
for p in drift_dir.glob("*.png"): p.unlink()

pipe = StableDiffusionPipeline.from_pretrained(config.BASE_MODEL)
pipe.load_lora_weights(str(config.LORA_DIR))      # rank-16 LoRA
pipe = pipe.to(config.DEVICE)
pipe.safety_checker = None

for k in range(10):
    g = torch.Generator(device=config.DEVICE).manual_seed(k)
    img = pipe("a photo of a man, color photograph, sharp focus, natural daylight",
               negative_prompt="sepia, grainy, vintage, black and white, blurry, low quality",
               num_inference_steps=40, guidance_scale=7.0, generator=g).images[0]
    img.save(drift_dir / f"drift_{k:03d}.png")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print("drift-test images saved")

evaluate.evaluate(generated_dir=drift_dir, train_dir=config.TRAIN_DIR)

In [ ]:
print(pipe.scheduler)